In [0]:
#SOLID PRINCIPLE

In [0]:
from abc import ABC, abstractmethod
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    col, regexp_replace, trim, coalesce, lit,
    to_date, current_date, sum as _sum
)
from pyspark.sql.window import Window


class DataExtractor(ABC):
    @abstractmethod
    def extract(self) -> DataFrame:
        pass


class OrdersExtractor(DataExtractor):
    def __init__(self, spark: SparkSession):
        self.spark = spark

    def extract(self) -> DataFrame:
        orders_data = [
            ("O1001", "C001", "10-01-2023", 250.0),
            ("O1002", "C002", "12-01-2023", 300.0),
            ("O1003", "C004", "15-01-2023", None)
        ]
        return self.spark.createDataFrame(
            orders_data,
            ["OrderID", "CustomerID", "OrderDate", "Amount"]
        )


class CRMExtractor(DataExtractor):
    def __init__(self, spark: SparkSession):
        self.spark = spark

    def extract(self) -> DataFrame:
        crm_data = [
            ("C001", "John@Doe", "USA"),
            ("C002", "Jane#Smith", "UK"),
            ("C003", "Mike!Johnson", "Ireland")
        ]
        return self.spark.createDataFrame(
            crm_data,
            ["CustomerID", "Name", "Country"]
        )


class CustomerCleaner:
    def clean(self, crm_df: DataFrame) -> DataFrame:
        return crm_df.withColumn(
            "Name",
            trim(
                regexp_replace(
                    regexp_replace(col("Name"), r"(^[^A-Za-z0-9]+|[^A-Za-z0-9]+$)", ""),
                    r"[^A-Za-z0-9]+",
                    " "
                )
            )
        )


class OrdersTransformer:
    def transform(self, orders_df: DataFrame) -> DataFrame:
        return orders_df.withColumn(
            "OrderDate",
            to_date(col("OrderDate"), "dd-MM-yyyy")
        ).withColumn(
            "Amount",
            coalesce(col("Amount"), lit(0.0))
        )


class ValidationRule(ABC):
    @abstractmethod
    def apply(self, df: DataFrame) -> DataFrame:
        pass


class OrderDateNotFutureRule(ValidationRule):
    def apply(self, df: DataFrame) -> DataFrame:
        return df.filter(col("OrderDate") <= current_date())


class CustomerExistsRule(ValidationRule):
    def apply(self, df: DataFrame) -> DataFrame:
        return df.filter(col("crm_customer_id").isNotNull())


class Validator:
    def __init__(self, rules: list[ValidationRule]):
        self.rules = rules

    def validate(self, df: DataFrame) -> DataFrame:
        result = df
        for rule in self.rules:
            result = rule.apply(result)
        return result


class OrderJoiner:
    def join_with_crm(self, orders_df: DataFrame, crm_df: DataFrame) -> DataFrame:
        crm_ids = crm_df.select(col("CustomerID").alias("crm_customer_id"))
        return orders_df.join(
            crm_ids,
            orders_df.CustomerID == crm_ids.crm_customer_id,
            "left"
        )


class ErrorLogger:
    def get_unmatched_customers(self, joined_df: DataFrame) -> DataFrame:
        return joined_df.filter(
            col("crm_customer_id").isNull()
        ).select(
            "OrderID", "CustomerID", "OrderDate", "Amount"
        )


class RunningTotalCalculator:
    def calculate(self, valid_orders: DataFrame) -> DataFrame:
        window_spec = Window.partitionBy("CustomerID").orderBy("OrderDate", "OrderID")
        return valid_orders.withColumn(
            "running_total_amount",
            _sum("Amount").over(window_spec)
        )


class AuditLogger:
    def __init__(self):
        self.rows = []

    def log(self, step: str, before_df: DataFrame | None, after_df: DataFrame | None):
        before_count = before_df.count() if before_df is not None else 0
        after_count = after_df.count() if after_df is not None else 0
        self.rows.append((step, before_count, after_count))

    def to_dataframe(self, spark: SparkSession) -> DataFrame:
        return spark.createDataFrame(
            self.rows,
            ["step_name", "before_count", "after_count"]
        )


class ETLPipeline:
    def __init__(
        self,
        orders_extractor: DataExtractor,
        crm_extractor: DataExtractor,
        customer_cleaner: CustomerCleaner,
        orders_transformer: OrdersTransformer,
        order_joiner: OrderJoiner,
        validator: Validator,
        error_logger: ErrorLogger,
        running_total_calculator: RunningTotalCalculator,
        audit_logger: AuditLogger,
        spark: SparkSession
    ):
        self.orders_extractor = orders_extractor
        self.crm_extractor = crm_extractor
        self.customer_cleaner = customer_cleaner
        self.orders_transformer = orders_transformer
        self.order_joiner = order_joiner
        self.validator = validator
        self.error_logger = error_logger
        self.running_total_calculator = running_total_calculator
        self.audit_logger = audit_logger
        self.spark = spark

    def run(self):
        orders_df = self.orders_extractor.extract()
        crm_df = self.crm_extractor.extract()
        self.audit_logger.log("extract_orders", None, orders_df)
        self.audit_logger.log("extract_crm", None, crm_df)

        crm_clean = self.customer_cleaner.clean(crm_df)
        self.audit_logger.log("clean_customer_names", crm_df, crm_clean)

        orders_clean = self.orders_transformer.transform(orders_df)
        self.audit_logger.log("prepare_orders", orders_df, orders_clean)

        joined_df = self.order_joiner.join_with_crm(orders_clean, crm_clean)
        self.audit_logger.log("join_orders_crm", orders_clean, joined_df)

        error_df = self.error_logger.get_unmatched_customers(joined_df)
        self.audit_logger.log("log_unmatched_customers", None, error_df)

        valid_orders = self.validator.validate(joined_df).select(
            "OrderID", "CustomerID", "OrderDate", "Amount"
        )
        self.audit_logger.log("validate_orders", joined_df, valid_orders)

        running_total_df = self.running_total_calculator.calculate(valid_orders)
        self.audit_logger.log("calculate_running_total", valid_orders, running_total_df)

        audit_df = self.audit_logger.to_dataframe(self.spark)

        return {
            "orders_df": orders_df,
            "crm_df": crm_df,
            "crm_clean": crm_clean,
            "orders_clean": orders_clean,
            "joined_df": joined_df,
            "error_df": error_df,
            "valid_orders": valid_orders,
            "running_total_df": running_total_df,
            "audit_df": audit_df
        }


spark = SparkSession.builder.appName("solid_etl").getOrCreate()

audit_logger = AuditLogger()

pipeline = ETLPipeline(
    orders_extractor=OrdersExtractor(spark),
    crm_extractor=CRMExtractor(spark),
    customer_cleaner=CustomerCleaner(),
    orders_transformer=OrdersTransformer(),
    order_joiner=OrderJoiner(),
    validator=Validator([
        OrderDateNotFutureRule(),
        CustomerExistsRule()
    ]),
    error_logger=ErrorLogger(),
    running_total_calculator=RunningTotalCalculator(),
    audit_logger=audit_logger,
    spark=spark
)

result = pipeline.run()

result["crm_clean"].show()
result["valid_orders"].show()
result["error_df"].show()
result["running_total_df"].show()
result["audit_df"].show()

+----------+------------+-------+
|CustomerID|        Name|Country|
+----------+------------+-------+
|      C001|    John Doe|    USA|
|      C002|  Jane Smith|     UK|
|      C003|Mike Johnson|Ireland|
+----------+------------+-------+

+-------+----------+----------+------+
|OrderID|CustomerID| OrderDate|Amount|
+-------+----------+----------+------+
|  O1001|      C001|2023-01-10| 250.0|
|  O1002|      C002|2023-01-12| 300.0|
+-------+----------+----------+------+

+-------+----------+----------+------+
|OrderID|CustomerID| OrderDate|Amount|
+-------+----------+----------+------+
|  O1003|      C004|2023-01-15|   0.0|
+-------+----------+----------+------+

+-------+----------+----------+------+--------------------+
|OrderID|CustomerID| OrderDate|Amount|running_total_amount|
+-------+----------+----------+------+--------------------+
|  O1001|      C001|2023-01-10| 250.0|               250.0|
|  O1002|      C002|2023-01-12| 300.0|               300.0|
+-------+----------+----------